# ONNX ????

ONNX(Open Neural Network Exchange)??????????????,????????????????????ONNX ?????????????????????,???????????????????????????????????? (ChatGPT 4o)

GitHub: [onnx/onnx](https://github.com/onnx/onnx)

????? titanic ???????? MLP ??,?????? ONNX ???

> **Note:** ??????,? torch.onnx.export ?????,?????????????????,?? batch_size, sequence_length ?????????,???????,????? Triton ? config.pbtxt ??????????



In [ ]:
import torch
import numpy as np
import onnxruntime as ort

import mlp
import utils

from sklearn.preprocessing import LabelEncoder

In [ ]:
DATA_PATH = 'data'
TITANIC_FN = 'train_and_test2.csv'
MLP_ONNX_FN = 'mlp.onnx'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 1. ????

??????????????? MLP ? [titanic](https://www.kaggle.com/datasets/heptapod/titanic/data) ?????? 20 ? epochs ??????????????

In [ ]:
csv_path = utils.gen_abspath(DATA_PATH, TITANIC_FN)
df = utils.read_csv(csv_path)
df

In [ ]:
# df.info()

### 1)???

In [ ]:
# ?????????
label_encoder = LabelEncoder()
df['new_embarked'] = label_encoder.fit_transform(df['Embarked'])

# ??????????
features = df[['Age', 'Fare', 'Sex', 'sibsp', 'Parch', 'Pclass', 'new_embarked']].values
features_tensor = torch.tensor(features, dtype=torch.float32)

df['embedding'] = [e for e in df[['Age', 'Fare', 'Sex', 'sibsp', 'Parch', 'Pclass', 'new_embarked']].values]
# df[['embedding', '2urvived']]

### 2)????

In [ ]:
# ????
mlp_model = mlp.SimpleMLP(
    input_channel=df.iloc[0]['embedding'].shape[0],
    output_channel=2,
    hidden_layers=[64, 32],
    dropout_probs=[0.4, 0.2],
    batch_size=32,
    num_epochs=20,
    ylim=[0.0, 1.0],
    test_size=0.2,
    random_state=31
)

predict, metrics = mlp_model(
    df=df,
    embd_col_name='embedding',
    label_col_name='2urvived'
)

print(f'train_loss: {metrics['train_loss']:.4f}')
print(f'train_acc: {metrics['train_acc']:.4f}')
print(f'test_acc: {metrics['test_acc']:.4f}')

In [ ]:
df['predict'] = predict
mask = df['2urvived'] != df['predict']
df[mask][['embedding', '2urvived', 'predict']]

### 3)?? torch.nn.Module

In [ ]:
net = mlp.MLP(
    input_channel=df.iloc[0]['embedding'].shape[0],
    output_channel=2,
    hidden_layers=[64, 32],
    dropout_probs=[0.4, 0.2]
)

tm_model = mlp.TrainMLP(net=net)
predict, metrics = tm_model(
    df=df,
    embd_col_name='embedding',
    label_col_name='2urvived'
)

print(f'train_loss: {metrics['train_loss']:.4f}')
print(f'train_acc: {metrics['train_acc']:.4f}')
print(f'test_acc: {metrics['test_acc']:.4f}')

In [ ]:
df['predict'] = predict
mask = df['2urvived'] != df['predict']
df[mask][['embedding', '2urvived', 'predict']]

In [ ]:
tm_model.net

## 2. ?? ONNX

In [ ]:
# ??????
features = df['embedding'].iloc[0].astype(np.float32)
tensor_features = torch.tensor(features).unsqueeze(0).to(device)
example_inputs = (tensor_features, )
example_inputs

In [ ]:
# mlp onnx ??????
mlp_onnx_path = utils.gen_abspath(DATA_PATH, MLP_ONNX_FN)

# ?? MLP ??
torch.onnx.export(
    model=tm_model.net,  # ??
    args=example_inputs,  # ??????
    f=mlp_onnx_path,  # ?????
    export_params=True,  # ??????
    opset_version=20,  # ONNX ?????
    input_names=['x'],  # ???
    output_names=['predict'],  # ???
    dynamic_axes={
        'x': {0: 'batch_size'},  # ? 0 ?????(batch_size)
        'predict': {0: 'batch_size'}
    }
)

In [ ]:
# ?? onnx ??
mlp_session = ort.InferenceSession(mlp_onnx_path)
outputs = mlp_session.run(None, {
    "x": tensor_features.cpu().numpy()
})
outputs